In [169]:
import os 
cwd = os.getcwd()
cwd

os.chdir("/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Datasets")

new_cwd = os.getcwd() 

new_cwd

'/Users/jmthomas565/Desktop/Education/Applied Stats MSc/MSc Project/Datasets'

In [170]:

import pandas as pd
import numpy as np
import plotly.express as px
from numpy import mean, std, absolute
from sklearn import metrics
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted, check_array
import nevergrad as ng
import statsmodels.tsa.api as tsa
import matplotlib.pyplot as plt 
import matplotlib.dates as mdates
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import TimeSeriesSplit
import warnings
from datetime import datetime, timedelta

In [171]:
bike_sales_data = pd.read_csv("bike_sales_data.csv")

base_rate_usa = pd.read_csv("base_rate_usa.csv")

gdp_usa = pd.read_csv("gdp_usa.csv")

unemployment_rate_usa = pd.read_csv("unemployment_rate_usa.csv")

### Cleansing Bike Sales Data

In [173]:

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    bike_sales_data = bike_sales_data.set_index("Week")
        
    bike_sales_data.index = pd.to_datetime(bike_sales_data.index)
    
    bike_sales_data.index.dtype
    
    bike_sales_data.head()


### Macroeconomic Variable Clean

In [175]:
#Setting the dates to the same as the bike sales data. Also formatting the dates


def index_to_date(df, col):
    
    df = df.set_index(col) 
    df.index = pd.to_datetime(df.index)
    df = df.loc['2017-07-01':'2022-10-01']
    
    return(df)


In [176]:
#Running the function 

base_rate_usa = index_to_date(base_rate_usa, 'observation_date')

gdp_usa = index_to_date(gdp_usa, 'observation_date')

unemployment_rate_usa = index_to_date(unemployment_rate_usa, 'observation_date')


In [177]:
#This code creates a list of dates. These will then be used for the left join, so that all dates are maintained.
#This is required, because the macroeconomic vars have different frequencies. 


start_date_string = '2017-07-01'
end_date_string = '2022-10-01'

start_date = datetime.strptime (start_date_string, '%Y-%m-%d')
end_date = datetime.strptime(end_date_string, '%Y-%m-%d')

date_list = [
    (start_date + timedelta (days=x)).strftime('%Y-%m-%d')
    for x in range((end_date - start_date).days + 1)
]

print(date_list[:5], '...', date_list[-5:])


['2017-07-01', '2017-07-02', '2017-07-03', '2017-07-04', '2017-07-05'] ... ['2022-09-27', '2022-09-28', '2022-09-29', '2022-09-30', '2022-10-01']


In [178]:
date_list = pd.DataFrame(date_list)

date_list[0] = pd.to_datetime(date_list[0])

date_list = date_list.set_index(0)

In [179]:

joined_df = date_list.join(gdp_usa['NA000334Q'], how='left')

joined_df = joined_df.join(base_rate_usa['FEDFUNDS'], how='left')

joined_df = joined_df.join(unemployment_rate_usa['UNRATENSA'], how='left')



In [180]:
#forward_fill is required, due to the macroeconomic variables having different time frequencies.

def forward_fill(df, joined_df):

    daily_index = pd.date_range (start=df.index.min(), end=df.index.max(), freq='D')
    
    df_daily = df.reindex (daily_index, method='ffill')
    
    #joining to the origional df
    joined_df_ff = joined_df.join (df_daily, how='left', lsuffix='_orig', rsuffix='_daily')

    return joined_df_ff


In [181]:
joined_df = forward_fill(gdp_usa, joined_df)

joined_df = forward_fill(base_rate_usa, joined_df)

joined_df = forward_fill(unemployment_rate_usa, joined_df)


In [182]:
joined_df = joined_df.drop(['NA000334Q_orig', 'FEDFUNDS_orig', 'UNRATENSA_orig'], axis = 1)

bike_sales_data = bike_sales_data.join(joined_df, how='left')

bike_sales_data = bike_sales_data.rename(columns={'NA000334Q_daily': 'gdp', 'FEDFUNDS_daily': 'base_rate', 'unemployment_rate_usa': 'UNRATENSA_daily'})


### Train-Test Split

In [184]:
Train = bike_sales_data.iloc[:208]

Test = bike_sales_data.iloc[208:]

In [185]:
X_train = Train.drop('sales', axis = 1)

y_train = Train['sales'].to_frame()


X_test = Test.drop('sales', axis = 1)

y_test = Test['sales'].to_frame()
